In [1]:
import os
import json
from pathlib import Path
from typing import List, Dict

In [2]:
import gradio as gr
from dotenv import load_dotenv

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore

In [4]:
# ─────────────────────────────────────────────────────────────
# 1. Config
# ─────────────────────────────────────────────────────────────
load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "Falta GOOGLE_API_KEY en .env"

In [5]:
DOCS_PATH   = Path("data/company_documents.json")
QDRANT_PATH = Path("./qdrant_data")   # mismo path que usaste en el notebook
COLLECTION  = "company_docs_lc_v2"       # mismo nombre de colección del notebook

In [6]:
# ─────────────────────────────────────────────────────────────
# 2. Cargar documentos
# ─────────────────────────────────────────────────────────────
def load_documents(path: Path) -> List[Dict]:
    with path.open(encoding="utf-8") as f:
        docs = json.load(f)
    required = {"id", "title", "content"}
    for i, doc in enumerate(docs):
        missing = required - doc.keys()
        if missing:
            raise ValueError(f"Documento {i} le faltan campos: {missing}")
    return docs

In [7]:
# ─────────────────────────────────────────────────────────────
# 3. Cargar pipeline desde índice persistido
#    El notebook ya hizo el chunking + embedding + upsert.
#    Aquí solo abrimos el índice y construimos la chain encima.
# ─────────────────────────────────────────────────────────────
def load_pipeline(documents: List[Dict]):
    if not QDRANT_PATH.exists():
        raise FileNotFoundError(
            f"No se encontró el índice en '{QDRANT_PATH}'.\n"
            "Asegúrate de haber corrido el notebook con:\n"
            "  qdrant_client = QdrantClient(path='./qdrant_data')"
        )

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY,
    )

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        google_api_key=GOOGLE_API_KEY,
        temperature=0.1,
    )

    # Abrir el índice existente — sin crear colección ni reindexar
    qdrant_client = QdrantClient(path=str(QDRANT_PATH))
    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )

    # Contar puntos en el índice para el status
    n_chunks = qdrant_client.count(collection_name=COLLECTION).count

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    prompt = ChatPromptTemplate.from_template("""
Eres un asistente empresarial. Responde la pregunta basandote UNICAMENTE
en el contexto proporcionado. Si la informacion no esta en el contexto,
di explicitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:
""")

    def format_docs(docs: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{d.metadata.get('title', '')}]\n{d.page_content}"
            for d in docs
        )

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )

    rag_with_sources = RunnableParallel(
        answer=rag_chain,
        sources=retriever,
    )

    return rag_with_sources, [d["title"] for d in documents], n_chunks

In [8]:
# ─────────────────────────────────────────────────────────────
# 4. Inicializar al arrancar
# ─────────────────────────────────────────────────────────────
print("Cargando índice desde disco...")
company_documents = load_documents(DOCS_PATH)
rag_pipeline, titles, n_chunks = load_pipeline(company_documents)
print(f"Listo — {len(titles)} docs, {n_chunks} chunks en el índice")

Cargando índice desde disco...
Listo — 4 docs, 5 chunks en el índice


In [15]:
# ─────────────────────────────────────────────────────────────
# 5. Logica del chat
# ─────────────────────────────────────────────────────────────
def chat(question: str, history: list):
    if not question.strip():
        return "", history, ""

    result  = rag_pipeline.invoke(question)
    answer  = result["answer"].strip()
    sources = result["sources"]

    if sources:
        seen = set()
        lines = ["**Fuentes recuperadas:**"]
        for doc in sources:
            t = doc.metadata.get("title", "—")
            if t not in seen:
                lines.append(f"- {t}")
                seen.add(t)
        lines.append(f"\n*{len(sources)} chunk(s)*")
        sources_md = "\n".join(lines)
    else:
        sources_md = "No se encontraron documentos relevantes."

    # Gradio 6 — formato de mensajes
    history.append({"role": "user",      "content": question})
    history.append({"role": "assistant", "content": answer})

    return "", history, sources_md

In [16]:
def clear_chat():
    return [], ""

In [19]:
# ─────────────────────────────────────────────────────────────
# 6. UI Gradio
# ─────────────────────────────────────────────────────────────
with gr.Blocks(title="RAG Empresarial") as demo:

    gr.Markdown("# RAG Empresarial\nConsulta los documentos internos usando Retrieval-Augmented Generation.")

    with gr.Row():

        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {t}" for t in titles))
            gr.Markdown(f"*{n_chunks} chunks en el indice*")
            gr.Markdown("---")
            gr.Markdown("### Pipeline")
            gr.Markdown(
                "- **Embeddings:** gemini-embedding-001\n"
                "- **LLM:** gemini-2.5-flash-lite\n"
                "- **Vector store:** Qdrant in-memory\n"
                "- **k=3, threshold=0.3**"
            )

        with gr.Column(scale=2):
            gr.Markdown("### Chat")

            chatbot = gr.Chatbot(height=420)

            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Haz una pregunta sobre los documentos...",
                    label="Pregunta",
                    lines=2,
                    scale=4,
                )
                send_btn = gr.Button("Enviar", variant="primary", scale=1)

            sources_box = gr.Markdown()
            clear_btn   = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["Cual es el precio del plan Enterprise de CloudSync Pro?"],
            ["Cuantos dias a la semana pueden trabajar remotamente los empleados?"],
            ["Que pasa con los reembolsos mayores a $100?"],
            ["Que ocurre durante la primera semana de onboarding?"],
        ],
        inputs=question_box,
        label="Ejemplos",
    )

    # Eventos
    send_btn.click(chat, [question_box, chatbot], [question_box, chatbot, sources_box])
    question_box.submit(chat, [question_box, chatbot], [question_box, chatbot, sources_box])
    clear_btn.click(clear_chat, outputs=[chatbot, sources_box])

In [26]:
# ─────────────────────────────────────────────────────────────
# 7. Launch
# ─────────────────────────────────────────────────────────────
demo.launch(
    server_port=8081,
    share=False,    # share=True para tunnel publico en Colab/LightningAI
    show_error=True,
)

* Running on local URL:  http://127.0.0.1:8081
* To create a public link, set `share=True` in `launch()`.


In [27]:
gr.close_all()

Closing server running on port: 8081
